# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import os
while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
    os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: /Users/ghaidaa/Desktop/flyrank-internship-ml


In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...):  ········


Connected.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

One row = one content item (a page), for one client, on one day — the natural grain of fact_content_daily_performance. For my feature frame, I aggregate these daily rows into one row per content item over a defined window.


imp_prior30, pos_prior30clk_prior30 : All are knowable strictly before the outcome window closes.

built from comparing imp_recent30 to imp_prior30 (>20% drop) : An observed outcome across a later window, not a same-window bucket.

client_hash_id, content_hash_id : Join/grouping keys only

trend_direction, trend_pct, any health_score style column 


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the stated grain (should be empty): {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be empty): 0


,client_hash_id,content_hash_id,report_date,n


In [ ]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the stated grain (should be empty): {len(grain_check)}")
grain_check

In [ ]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

print(slice_stats)

In [ ]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

print(availability)
print(f"\nShare of rows with real GA4 data: "
      f"{availability['ga4_available_rows'][0] / availability['total_rows'][0]:.1%}")

In [ ]:
window_check = con.sql(f"""
    WITH bounds AS (SELECT DATE '2026-03-31' AS anchor_d),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date > b.anchor_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_recent30,
               SUM(CASE WHEN f.report_date <= b.anchor_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prior30,
               AVG(CASE WHEN f.report_date <= b.anchor_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prior30,
               SUM(CASE WHEN f.report_date <= b.anchor_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prior30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.anchor_d - INTERVAL 60 DAY AND f.report_date <= b.anchor_d
        GROUP BY 1, 2
        HAVING imp_prior30 >= 50
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

feat = window_check.merge(qsignals, on='content_hash_id', how='left')
print(f"{len(feat):,} content items in the feature frame")
feat.head()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

feat['is_declining'] = (feat['imp_recent30'] < 0.8 * feat['imp_prior30']).astype(int)

# LEAKY: add a label-derived column on purpose
feat['leak_pct_change'] = ((feat['imp_recent30'] - feat['imp_prior30'])
                            / feat['imp_prior30'].replace(0, np.nan)).fillna(0)

leaky_cols = ['imp_prior30', 'pos_prior30', 'clk_prior30', 'visible_queries', 'leak_pct_change']
X_leaky = feat[leaky_cols].fillna(0)
y = feat['is_declining']

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])
print(f"LEAKY ROC-AUC (with leak_pct_change): {leaky_auc:.3f}  <- suspiciously close to perfect")

# HONEST: remove the leaked column
honest_cols = ['imp_prior30', 'pos_prior30', 'clk_prior30', 'visible_queries']
X_honest = feat[honest_cols].fillna(0)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print(f"HONEST ROC-AUC (leak removed): {honest_auc:.3f}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

(1) whether a pattern found in March  generalizes to other months  the panel is unbalanced, with wildly different history depth per client, so March's client mix isn't guaranteed representative (2) anything about clients whose GSC/GA4 tracking started after March(3) causal reasons for decline , this data shows what happened, not why 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.